# Logan River distributed NGEN current-run evaluation

Evaluate and visualize the current `workshop_run_distributed_asyncdds` NGEN run while it is still running or after it completes. The notebook reads the generated NGEN output CSVs directly, detects the outlet nexus from the generated hydrofabric, aligns the simulated outlet discharge with USGS observations, and computes provisional metrics over the completed portion of the run.

The metrics and plots are diagnostic while NGEN is still writing output files.


In [ ]:
# Imports and paths
from pathlib import Path
import json
import warnings

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

warnings.filterwarnings("ignore", message=".*is an EXPERIMENTAL module.*")

repo_dir = Path.cwd()
if repo_dir.name == ".ipynb_checkpoints":
    repo_dir = repo_dir.parent

config_path = repo_dir / "config_logan_river_distributed_asyncdds.yaml"
if not config_path.exists():
    config_path = Path("examples/04_workshop_notebooks/config_logan_river_distributed_asyncdds.yaml")

with config_path.open() as handle:
    config = yaml.safe_load(handle)

data_dir = Path(config["SYMFLUENCE_DATA_DIR"])
domain_name = config["DOMAIN_NAME"]
experiment_id = config["EXPERIMENT_ID"]
project_dir = data_dir / f"domain_{domain_name}"
settings_dir = project_dir / "settings" / "NGEN"
run_dir = project_dir / "simulations" / experiment_id / "NGEN"
obs_path = (
    project_dir
    / "data"
    / "observations"
    / "streamflow"
    / "preprocessed"
    / f"{domain_name}_streamflow_processed.csv"
)

print(f"Config: {config_path}")
print(f"Project: {project_dir}")
print(f"Run dir: {run_dir}")
print(f"Observations: {obs_path}")


In [ ]:
# Detect the outlet nexus from the generated routing hydrofabric
def detect_outlet_nexus(settings_dir: Path) -> str:
    troute_gpkg = settings_dir / f"{domain_name}_hydrofabric_troute.gpkg"
    nexus_geojson = settings_dir / "nexus.geojson"

    if troute_gpkg.exists():
        flowpaths = gpd.read_file(troute_gpkg, layer="flowpaths")
        nexus = gpd.read_file(troute_gpkg, layer="nexus")
        flowpath_ids = set(flowpaths["id"].astype(str))
        outlet_flowpaths = flowpaths[
            flowpaths["toid"].isna()
            | ~flowpaths["toid"].astype(str).isin(flowpath_ids)
        ]
        if len(outlet_flowpaths) == 1:
            outlet_wb = str(outlet_flowpaths.iloc[0]["id"])
            outlet_nex = outlet_wb.replace("wb-", "nex-")
            if outlet_nex in set(nexus["id"].astype(str)):
                return outlet_nex

    if nexus_geojson.exists():
        data = json.loads(nexus_geojson.read_text())
        ids = [feature.get("id") for feature in data.get("features", [])]
        if "nex-341" in ids:
            return "nex-341"
        if ids:
            return sorted(ids)[-1]

    raise FileNotFoundError("Could not detect outlet nexus from NGEN settings")

outlet_nexus_id = detect_outlet_nexus(settings_dir)
outlet_output_path = run_dir / f"{outlet_nexus_id}_output.csv"

print(f"Outlet nexus: {outlet_nexus_id}")
print(f"Outlet output: {outlet_output_path}")


In [ ]:
# Read NGEN output safely, even while the file is still being written
def read_ngen_output(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Missing NGEN output: {path}")

    records = []
    with path.open("r", encoding="utf-8", errors="ignore") as handle:
        for line in handle:
            parts = [part.strip() for part in line.split(",")]
            if len(parts) < 3:
                continue
            try:
                step = int(parts[0])
                timestamp = pd.to_datetime(parts[1])
                discharge = float(parts[2])
            except (TypeError, ValueError):
                continue
            records.append((step, timestamp, discharge))

    if not records:
        raise ValueError(f"No complete rows found in {path}")

    df = pd.DataFrame(records, columns=["step", "datetime", "sim_discharge_cms"])
    return df.drop_duplicates("datetime").set_index("datetime").sort_index()

sim = read_ngen_output(outlet_output_path)

start_time = pd.to_datetime(config["EXPERIMENT_TIME_START"])
end_time = pd.to_datetime(config["EXPERIMENT_TIME_END"])
forcing_step_seconds = int(config.get("FORCING_TIME_STEP_SIZE", 3600))
expected_steps = int((end_time - start_time).total_seconds() // forcing_step_seconds) + 1
current_step = int(sim["step"].max())
current_time = sim.index.max()

print(f"Rows read: {len(sim)}")
print(f"Current timestep: {current_step + 1}/{expected_steps}")
print(f"Current model time: {current_time}")
print(f"Progress: {(current_step + 1) / expected_steps:.1%}")


In [ ]:
# Read and align USGS observations
obs = pd.read_csv(obs_path)
obs["datetime"] = pd.to_datetime(obs["datetime"], utc=True, errors="coerce").dt.tz_convert(None)
obs = obs.dropna(subset=["datetime"]).set_index("datetime").sort_index()

obs_hourly = obs["discharge_cms"].resample("h").mean()
common_index = sim.index.intersection(obs_hourly.index)

aligned = pd.DataFrame(
    {
        "observed_cms": obs_hourly.loc[common_index],
        "simulated_cms": sim.loc[common_index, "sim_discharge_cms"],
    }
).dropna()

spinup_end = pd.to_datetime(config["SPINUP_PERIOD"].split(",")[1].strip())
eval_aligned = aligned.loc[aligned.index > spinup_end]

print(f"Aligned rows including spinup: {len(aligned)}")
print(f"Aligned rows after spinup: {len(eval_aligned)}")
print(f"Evaluation window available: {eval_aligned.index.min()} to {eval_aligned.index.max()}")


In [ ]:
# Hydrologic metrics over the completed post-spinup period
def nse(obs_values: pd.Series, sim_values: pd.Series) -> float:
    denominator = ((obs_values - obs_values.mean()) ** 2).sum()
    if denominator == 0:
        return np.nan
    return 1.0 - ((sim_values - obs_values) ** 2).sum() / denominator


def kge(obs_values: pd.Series, sim_values: pd.Series) -> tuple[float, float, float, float]:
    if len(obs_values) < 2:
        return np.nan, np.nan, np.nan, np.nan
    r = np.corrcoef(obs_values, sim_values)[0, 1]
    alpha = sim_values.std(ddof=1) / obs_values.std(ddof=1)
    beta = sim_values.mean() / obs_values.mean()
    value = 1.0 - np.sqrt((r - 1.0) ** 2 + (alpha - 1.0) ** 2 + (beta - 1.0) ** 2)
    return value, r, alpha, beta


def percent_bias(obs_values: pd.Series, sim_values: pd.Series) -> float:
    return 100.0 * (sim_values.sum() - obs_values.sum()) / obs_values.sum()

obs_eval = eval_aligned["observed_cms"]
sim_eval = eval_aligned["simulated_cms"]
kge_value, corr, variability_ratio, bias_ratio = kge(obs_eval, sim_eval)
metrics = {
    "NSE": nse(obs_eval, sim_eval),
    "KGE": kge_value,
    "Correlation": corr,
    "Variability ratio": variability_ratio,
    "Bias ratio": bias_ratio,
    "PBIAS (%)": percent_bias(obs_eval, sim_eval),
    "RMSE (cms)": np.sqrt(((sim_eval - obs_eval) ** 2).mean()),
}

pd.Series(metrics).to_frame("value")


In [ ]:
# Streamflow diagnostic plots
plot_data = eval_aligned.copy()
if plot_data.empty:
    raise ValueError("No post-spinup overlap is available yet")

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle(
    f"Logan River distributed NGEN partial evaluation - {outlet_nexus_id}\n"
    f"through {plot_data.index.max()}",
    fontsize=14,
    fontweight="bold",
)

# Time series
axes[0, 0].plot(plot_data.index, plot_data["observed_cms"], label="Observed (USGS)", linewidth=1.0)
axes[0, 0].plot(plot_data.index, plot_data["simulated_cms"], label="NGEN", linewidth=1.0)
axes[0, 0].set_title("Hourly streamflow")
axes[0, 0].set_ylabel("Discharge (m3/s)")
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)
axes[0, 0].text(
    0.02,
    0.95,
    f"NSE: {metrics['NSE']:.3f}\n"
    f"KGE: {metrics['KGE']:.3f}\n"
    f"Bias: {metrics['PBIAS (%)']:.1f}%",
    transform=axes[0, 0].transAxes,
    va="top",
    bbox={"facecolor": "white", "alpha": 0.85},
)

# Scatter
axes[0, 1].scatter(plot_data["observed_cms"], plot_data["simulated_cms"], s=8, alpha=0.35)
max_flow = max(plot_data["observed_cms"].max(), plot_data["simulated_cms"].max())
axes[0, 1].plot([0, max_flow], [0, max_flow], "k--", alpha=0.5)
axes[0, 1].set_title("Observed vs simulated")
axes[0, 1].set_xlabel("Observed (m3/s)")
axes[0, 1].set_ylabel("Simulated (m3/s)")
axes[0, 1].grid(alpha=0.3)

# Monthly means for available period
monthly = plot_data.groupby(plot_data.index.month).mean(numeric_only=True)
axes[1, 0].plot(monthly.index, monthly["observed_cms"], "o-", label="Observed")
axes[1, 0].plot(monthly.index, monthly["simulated_cms"], "o-", label="NGEN")
axes[1, 0].set_xticks(range(1, 13))
axes[1, 0].set_title("Monthly mean flow for completed period")
axes[1, 0].set_ylabel("Discharge (m3/s)")
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)

# Flow duration curve
obs_sorted = np.sort(plot_data["observed_cms"].to_numpy())[::-1]
sim_sorted = np.sort(plot_data["simulated_cms"].to_numpy())[::-1]
obs_exceedance = np.arange(1, len(obs_sorted) + 1) / (len(obs_sorted) + 1) * 100.0
sim_exceedance = np.arange(1, len(sim_sorted) + 1) / (len(sim_sorted) + 1) * 100.0
axes[1, 1].semilogy(obs_exceedance, obs_sorted, label="Observed")
axes[1, 1].semilogy(sim_exceedance, sim_sorted, label="NGEN")
axes[1, 1].set_title("Flow duration curve")
axes[1, 1].set_xlabel("Exceedance probability (%)")
axes[1, 1].set_ylabel("Discharge (m3/s)")
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# Progress across all nexus output files
def last_complete_row(path: Path) -> tuple[int, pd.Timestamp] | None:
    try:
        with path.open("rb") as handle:
            handle.seek(0, 2)
            end = handle.tell()
            data = b""
            block_size = 8192
            while end > 0:
                start = max(0, end - block_size)
                handle.seek(start)
                data = handle.read(end - start) + data
                if data.count(b"\n") >= 2 or start == 0:
                    break
                end = start
    except FileNotFoundError:
        return None

    for raw_line in reversed(data.splitlines()):
        parts = [part.strip() for part in raw_line.decode(errors="ignore").split(",")]
        if len(parts) < 3:
            continue
        try:
            return int(parts[0]), pd.to_datetime(parts[1])
        except (TypeError, ValueError):
            continue
    return None

rows = []
for path in sorted(run_dir.glob("nex-*output.csv")):
    last_row = last_complete_row(path)
    if last_row is None:
        continue
    step, timestamp = last_row
    rows.append((path.name, step, timestamp))

progress = pd.DataFrame(rows, columns=["file", "step", "time"])
if not progress.empty:
    print(f"Nexus files with output: {len(progress)}")
    print(f"Slowest: {progress['time'].min()} ({progress['step'].min() + 1}/{expected_steps})")
    print(f"Fastest: {progress['time'].max()} ({progress['step'].max() + 1}/{expected_steps})")
    print(f"Fastest progress: {(progress['step'].max() + 1) / expected_steps:.1%}")
    display(progress.sort_values("step").head())
    display(progress.sort_values("step").tail())
else:
    print("No nexus output files found yet")
